In [1]:
import tensorflow as tf
from tensorflow.python.ops.gradient_checker_v2 import max_error

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
  for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

import warnings
import os 
warnings.filterwarnings("ignore")
import config
from utils import *
import math 
from tensorflow.keras.callbacks import TerminateOnNaN, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from numba import cuda 

from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
tf.keras.backend.clear_session()

os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"




2025-03-02 12:31:03.547717: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1740915063.561746   28906 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1740915063.566275   28906 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-02 12:31:03.580059: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


{'Cargo plane': 635, 'Helicopter': 70, 'Small car': 4290, 'Bus': 2155, 'Truck': 2746, 'Motorboat': 1069, 'Fishing vessel': 706, 'Dump truck': 1236, 'Excavator': 789, 'Building': 4689, 'Storage tank': 1469, 'Shipping container': 1523}


I0000 00:00:1740915066.742238   28906 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6717 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1070, pci bus id: 0000:02:00.0, compute capability: 6.1


In [5]:
from keras_tuner import Hyperband

# Run the hyperparameter search
batch_size = 64
train_generator, valid_generator, sz_train, sz_val = train_val_split(batch_size=batch_size)
train_steps = math.ceil(sz_train / batch_size)
valid_steps = math.ceil(sz_val / batch_size)


# Set up Hyperband
tuner = Hyperband(
    build_fcnn,
    objective='val_accuracy',
    max_epochs=80,
    factor=10,
    directory='hyperband_search',
    project_name='fcnn_tuning',
    max_model_size=1_000_000_000,
    max_consecutive_failed_trials=5,
    executions_per_trial=1  # Disallow parallel execution
)

# Define callback for the search
early_stop_tuner = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)

# hyperband_checkpoint = HyperbandCheckpointCallback(tuner)
class ClearGPUCache(Callback):
    def on_epoch_end(self, epoch, logs=None):
        print(f"Clearing GPU cache after epoch {epoch + 1}")
        tf.keras.backend.clear_session()  # Clears TensorFlow session
    
    def on_train_end(self, logs=None):
        print("Ending training")
        tf.keras.backend.clear_session()  # Clears TensorFlow session
        
tuner.search(
    train_generator,
    steps_per_epoch=train_steps,
    validation_data=valid_generator,
    validation_steps=valid_steps,
    epochs=100,
    callbacks=[early_stop_tuner, ClearGPUCache()]
)

# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best hyperparameters found:")
for param, value in best_hps.values.items():
    print(f"{param}: {value}")



Trial 16 Complete [00h 00m 10s]

Best val_accuracy So Far: 0.3685687482357025
Total elapsed time: 02h 32m 28s

Search: Running Trial #17

Value             |Best Value So Far |Hyperparameter
3                 |3                 |num_layers
512               |768               |neurons_0
0.0012926         |0.00050051        |l1_0
0.012564          |0.00033594        |l2_0
leaky_relu        |elu               |activation_0
he                |he                |init_0
True              |True              |batch_norm_0
0.2               |0.1               |dropout_0
896               |640               |neurons_1
0.0021133         |0.0044252         |l1_1
0.0026882         |0.00061495        |l2_1
leaky_relu        |elu               |activation_1
he                |glorot            |init_1
True              |False             |batch_norm_1
0.4               |0.4               |dropout_1
768               |640               |neurons_2
0.0058105         |0.0085948         |l1_2
0.077661   

2025-03-01 22:15:56.048668: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:497] Allocator (GPU_0_bfc) ran out of memory trying to allocate 294.00MiB (rounded to 308281344)requested by op AddV2
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-03-01 22:15:56.049174: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1053] BFCAllocator dump for GPU_0_bfc
2025-03-01 22:15:56.049235: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1060] Bin (256): 	Total Chunks: 6726, Chunks in use: 6726. 1.64MiB allocated for chunks. 1.64MiB in use in bin. 27.6KiB client-requested in use in bin.
2025-03-01 22:15:56.049259: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1060] Bin (512): 	Total Chunks: 1, Chunks in use: 1. 512B allocated for chunks. 512B in use in bin. 400B client-requested in use in bin.
2025-03-01 22

RuntimeError: Number of consecutive failures exceeded the limit of 5.
Traceback (most recent call last):
  File "/home/fran/.virtualenvs/DeepLearning/lib/python3.12/site-packages/keras_tuner/src/engine/base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "/home/fran/.virtualenvs/DeepLearning/lib/python3.12/site-packages/keras_tuner/src/engine/base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/fran/.virtualenvs/DeepLearning/lib/python3.12/site-packages/keras_tuner/src/tuners/hyperband.py", line 427, in run_trial
    return super().run_trial(trial, *fit_args, **fit_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/fran/.virtualenvs/DeepLearning/lib/python3.12/site-packages/keras_tuner/src/engine/tuner.py", line 314, in run_trial
    obj_value = self._

In [ ]:
from keras_tuner import BayesianOptimization
import gc

# Run the hyperparameter search
batch_size = 256
train_generator, valid_generator, sz_train, sz_val = train_val_split(batch_size=batch_size)
train_steps = math.ceil(sz_train / batch_size)
valid_steps = math.ceil(sz_val / batch_size)

# Set up Bayesian Optimizer
tuner = BayesianOptimization(
    build_fcnn,
    objective='val_accuracy',
    max_trials=40,  # Number of total trials to run
    directory='bayesian_search',
    project_name='fcnn_tuning',
    overwrite=True,
    max_model_size=1_000_000_000,
    max_consecutive_failed_trials=5,
    executions_per_trial=1  # Disallow parallel execution
)

# Define callback for the search
early_stop_tuner = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)

class ClearGPUCache(Callback):
    def on_epoch_end(self, epoch, logs=None):
        print(f"Clearing GPU cache after epoch {epoch + 1}")
        gc.collect()
        tf.keras.backend.clear_session()  # Clears TensorFlow session
    
    def on_train_end(self, logs=None):
        print("Ending training")
        gc.collect()
        tf.keras.backend.clear_session()  # Clears TensorFlow session
        
tuner.search(
    train_generator,
    steps_per_epoch=train_steps,
    validation_data=valid_generator,
    validation_steps=valid_steps,
    epochs=30,
    callbacks=[early_stop_tuner, ClearGPUCache()]
)

# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]


Search: Running Trial #1

Value             |Best Value So Far |Hyperparameter
3                 |3                 |num_layers
704               |704               |neurons_0
0.00034604        |0.00034604        |l1_0
0.0014393         |0.0014393         |l2_0
elu               |elu               |activation_0
he                |he                |init_0
False             |False             |batch_norm_0
0.3               |0.3               |dropout_0
960               |960               |neurons_1
0.00013858        |0.00013858        |l1_1
0.080693          |0.080693          |l2_1
leaky_relu        |leaky_relu        |activation_1
he                |he                |init_1
False             |False             |batch_norm_1
0.1               |0.1               |dropout_1
640               |640               |neurons_2
0.00097317        |0.00097317        |l1_2
0.015652          |0.015652          |l2_2
leaky_relu        |leaky_relu        |activation_2
he                |he       

In [ ]:
from keras_tuner import BayesianOptimization

# Run the hyperparameter search
batch_size = 124
train_generator, valid_generator, sz_train, sz_val = train_val_split(batch_size=batch_size)
train_steps = math.ceil(sz_train / batch_size)
valid_steps = math.ceil(sz_val / batch_size)

# Set up Bayesian Optimizer
tuner = BayesianOptimization(
    build_fcnn,
    objective='val_accuracy',
    max_trials=40,  # Number of total trials to run
    directory='bayesian_search',
    project_name='fcnn_tuning',
    overwrite=True,
    max_model_size=1_000_000_000,
    max_consecutive_failed_trials=5,
    executions_per_trial=1  # Disallow parallel execution
)

# Define callback for the search
early_stop_tuner = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)

class ClearGPUCache(Callback):
    def on_epoch_end(self, epoch, logs=None):
        print(f"Clearing GPU cache after epoch {epoch + 1}")
        tf.keras.backend.clear_session()  # Clears TensorFlow session
    
    def on_train_end(self, logs=None):
        print("Ending training")
        tf.keras.backend.clear_session()  # Clears TensorFlow session
        
tuner.search(
    train_generator,
    steps_per_epoch=train_steps,
    validation_data=valid_generator,
    validation_steps=valid_steps,
    epochs=30,
    callbacks=[early_stop_tuner, ClearGPUCache()]
)

# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

In [ ]:
# Build the model with the best hyperparameters
best_model = tuner.hypermodel.build(best_hps)

# Define callbacks for the final model
model_checkpoint = ModelCheckpoint(
    'best_hyperband_model.keras',
    monitor='val_accuracy',
    verbose=1,
    save_best_only=True
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.1,
    patience=5,
    verbose=1
)
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=20,
    verbose=1,
    restore_best_weights=True
)
callbacks = [model_checkpoint, reduce_lr, early_stop]

# If you have custom callbacks, add them here
# callbacks = callbacks + [TimingCallback(), HistorySaverCallback()]


        
# Train the best model
history = best_model.fit(
    train_generator,
    steps_per_epoch=train_steps,
    validation_data=valid_generator,
    validation_steps=valid_steps,
    epochs=100,  # You can train longer since we have early stopping
    callbacks=callbacks,
    verbose=1,
)

# Best validation model
best_idx = int(np.argmax(history.history['val_accuracy']))
best_value = np.max(history.history['val_accuracy'])
print('Best validation model: epoch ' + str(best_idx+1), ' - val_accuracy ' + str(best_value))

# Save the best hyperparameters to a file for future reference
import json
with open('best_hyperparameters.json', 'w') as f:
    json.dump(best_hps.values, f, indent=2)